# Datathon Passos Mágicos — Fase 5  
## Notebook 2 revisado e comentado célula a célula  
### Modelo preditivo de risco de defasagem **sem vazamento de informação**

Este notebook substitui a versão original do **Notebook 2**.

## O que foi corrigido
Na versão original, o modelo usava indicadores do **mesmo ano** para prever a `defasagem` do **mesmo ano**.  
Isso gera um problema metodológico importante: algumas variáveis, principalmente `IAN`, ficam muito próximas da própria resposta que se quer prever.

Neste notebook, o desenho foi corrigido:

- usamos os indicadores do ano **t**
- para prever o risco de defasagem no ano **t+1**
- validamos o modelo de forma mais robusta
- regularizamos o Random Forest para reduzir risco de overfitting

## Objetivo analítico
Prever se um aluno estará em **risco de defasagem no ano seguinte** a partir dos indicadores observados no ano atual.

## Estrutura do notebook
1. Importação das bibliotecas  
2. Leitura e padronização dos dados  
3. Construção da base longitudinal  
4. Criação da variável-alvo futura  
5. Escolha das features  
6. Validação cruzada e teste temporal  
7. Treinamento final  
8. Interpretação e salvamento do modelo

## Célula 1 — Importação das bibliotecas

**Por que esta célula existe?**  
Ela reúne as bibliotecas necessárias para leitura, preparação, modelagem, avaliação e salvamento do modelo.

**O que mudou em relação ao notebook original?**
- adicionamos métricas mais completas;
- adicionamos validação cruzada estratificada;
- deixamos o notebook mais autônomo, com menos dependência de funções externas.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Célula 2 — Definição do caminho do arquivo

**Objetivo**  
Localizar a planilha Excel com os dados de 2022, 2023 e 2024.

**Comentário importante**  
Deixei mais de uma opção de caminho para facilitar o uso:
- dentro do projeto (`../data/...`)
- no mesmo diretório do notebook
- caminho absoluto, se necessário

In [ ]:
# Ajuste os caminhos abaixo conforme a sua estrutura de pastas.
caminhos_possiveis = [
    Path("../data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"),
    Path("./BASE DE DADOS PEDE 2024 - DATATHON.xlsx"),
    Path("/mnt/data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"),
]

DATA_PATH = None
for caminho in caminhos_possiveis:
    if caminho.exists():
        DATA_PATH = caminho
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Não foi possível localizar o arquivo Excel. Ajuste a lista 'caminhos_possiveis'."
    )

print(f"Arquivo localizado em: {DATA_PATH}")

## Célula 3 — Leitura das abas do Excel

**Objetivo**  
Ler separadamente as abas `PEDE2022`, `PEDE2023` e `PEDE2024`.

**Por que não carregar tudo de uma vez?**  
Porque os nomes das colunas mudam entre anos.  
Ao ler cada aba separadamente, fica mais fácil padronizar depois.

In [ ]:
df_2022 = pd.read_excel(DATA_PATH, sheet_name="PEDE2022")
df_2023 = pd.read_excel(DATA_PATH, sheet_name="PEDE2023")
df_2024 = pd.read_excel(DATA_PATH, sheet_name="PEDE2024")

print("Shapes originais:")
print("2022:", df_2022.shape)
print("2023:", df_2023.shape)
print("2024:", df_2024.shape)

## Célula 4 — Função de padronização dos nomes de colunas

**Problema que esta célula resolve**  
A base muda a grafia entre os anos. Exemplos:
- `INDE 22`, `INDE 2023`, `INDE 2024`
- `Defas`, `Defasagem`
- `Nome`, `Nome Anonimizado`
- `Idade 22`, `Idade`

Se não padronizarmos isso, não conseguimos montar uma base longitudinal consistente.

**O que esta função faz?**
- renomeia colunas equivalentes para um padrão comum;
- cria a coluna `ano`;
- mantém o restante da estrutura.

In [ ]:
def padronizar_ano(df: pd.DataFrame, ano: int) -> pd.DataFrame:
    df = df.copy()

    mapa = {}

    if ano == 2022:
        mapa = {
            "Nome": "nome",
            "Ano nasc": "ano_nasc",
            "Idade 22": "idade",
            "Instituição de ensino": "instituicao_ensino",
            "INDE 22": "inde",
            "IAA": "iaa",
            "IEG": "ieg",
            "IPS": "ips",
            "IDA": "ida",
            "IPV": "ipv",
            "IAN": "ian",
            "Defas": "defasagem",
            "Fase ideal": "fase_ideal",
            "Pedra 22": "pedra",
            "Matem": "mat",
            "Portug": "por",
            "Inglês": "ing",
            "Gênero": "genero",
        }
    elif ano == 2023:
        mapa = {
            "Nome Anonimizado": "nome",
            "Data de Nasc": "data_nasc",
            "Idade": "idade",
            "Instituição de ensino": "instituicao_ensino",
            "INDE 2023": "inde",
            "IAA": "iaa",
            "IEG": "ieg",
            "IPS": "ips",
            "IPP": "ipp",
            "IDA": "ida",
            "IPV": "ipv",
            "IAN": "ian",
            "Defasagem": "defasagem",
            "Fase Ideal": "fase_ideal",
            "Pedra 2023": "pedra",
            "Mat": "mat",
            "Por": "por",
            "Ing": "ing",
            "Gênero": "genero",
        }
    elif ano == 2024:
        mapa = {
            "Nome Anonimizado": "nome",
            "Data de Nasc": "data_nasc",
            "Idade": "idade",
            "Instituição de ensino": "instituicao_ensino",
            "INDE 2024": "inde",
            "IAA": "iaa",
            "IEG": "ieg",
            "IPS": "ips",
            "IPP": "ipp",
            "IDA": "ida",
            "IPV": "ipv",
            "IAN": "ian",
            "Defasagem": "defasagem",
            "Fase Ideal": "fase_ideal",
            "Pedra 2024": "pedra",
            "Mat": "mat",
            "Por": "por",
            "Ing": "ing",
            "Gênero": "genero",
            "Escola": "escola",
            "Ativo/ Inativo": "status_matricula",
        }

    df = df.rename(columns=mapa)
    df["ano"] = ano

    # Padronização leve dos nomes restantes para minúsculas e underscore
    df.columns = [
        str(col)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("º", "n")
        .replace("/", "_")
        .replace("-", "_")
        .replace(".", "")
        for col in df.columns
    ]
    return df

## Célula 5 — Aplicação da padronização

**Objetivo**  
Aplicar a função do passo anterior e verificar as colunas resultantes.

**Por que esta conferência é importante?**  
Porque o sucesso do modelo depende de conseguirmos:
- ligar o mesmo aluno ao longo dos anos;
- usar as mesmas features com o mesmo nome.

In [ ]:
df_2022_p = padronizar_ano(df_2022, 2022)
df_2023_p = padronizar_ano(df_2023, 2023)
df_2024_p = padronizar_ano(df_2024, 2024)

print("Exemplo de colunas padronizadas em 2022:")
print(df_2022_p.columns.tolist()[:30])

print("\nExemplo de colunas padronizadas em 2023:")
print(df_2023_p.columns.tolist()[:30])

print("\nExemplo de colunas padronizadas em 2024:")
print(df_2024_p.columns.tolist()[:30])

## Célula 6 — União das bases anuais

**Objetivo**  
Criar uma base única para inspeção geral.

**Observação**  
Ainda não é a base final do modelo.  
Ela serve para:
- entender cobertura;
- checar consistência;
- analisar quantos alunos aparecem em mais de um ano.

In [ ]:
df = pd.concat([df_2022_p, df_2023_p, df_2024_p], ignore_index=True, sort=False)

print("Shape da base unificada:", df.shape)
df[["ra", "ano", "inde", "ida", "ieg", "ips", "ipv", "ian", "defasagem"]].head()

## Célula 7 — Conferência da chave longitudinal (`RA`)

**Por que esta célula é central?**  
O desenho revisado depende de acompanhar o mesmo aluno em anos diferentes.  
Para isso, a chave `RA` precisa existir e se repetir ao longo do tempo.

**Se esta etapa falhar, o modelo longitudinal não funciona.**

In [ ]:
alunos_por_ano = df.groupby("ano")["ra"].nunique().rename("qtd_alunos")
print(alunos_por_ano)

ra_22 = set(df_2022_p["ra"].dropna().unique())
ra_23 = set(df_2023_p["ra"].dropna().unique())
ra_24 = set(df_2024_p["ra"].dropna().unique())

print("\nAlunos em 2022 e 2023:", len(ra_22.intersection(ra_23)))
print("Alunos em 2023 e 2024:", len(ra_23.intersection(ra_24)))

## Célula 8 — Criação do alvo futuro

**Mudança metodológica mais importante do notebook**

Na versão original, o modelo fazia isto:

- usava indicadores do ano `t`
- para prever a `defasagem` do próprio ano `t`

Isso é frágil porque o modelo enxerga variáveis quase contemporâneas ao próprio alvo.

### Novo desenho
Agora fazemos isto:

- pegamos as features do ano `t`
- e associamos à situação de risco no ano `t+1`

### Definição do alvo
`risco_defasagem_proximo_ano = 1` quando `defasagem_{t+1} <= -1`

In [ ]:
# Base de features do ano t
colunas_features_base = [
    "ra", "ano", "idade", "ano_ingresso",
    "cg", "cf", "ct",
    "iaa", "ieg", "ips", "ipp", "ida", "ipv", "ian", "inde",
    "mat", "por", "ing"
]

base_t = df[[c for c in colunas_features_base if c in df.columns]].copy()

# Base do alvo no ano t+1
alvo_t1 = df[["ra", "ano", "defasagem"]].copy()
alvo_t1["ano_origem"] = alvo_t1["ano"] - 1
alvo_t1["risco_defasagem_proximo_ano"] = (alvo_t1["defasagem"] <= -1).astype(int)

# Fazemos o merge:
# aluno no ano t  ---> alvo observado no ano t+1
base_modelo = base_t.merge(
    alvo_t1[["ra", "ano_origem", "risco_defasagem_proximo_ano"]],
    left_on=["ra", "ano"],
    right_on=["ra", "ano_origem"],
    how="inner"
)

base_modelo = base_modelo.drop(columns=["ano_origem"])

print("Shape da base longitudinal do modelo:", base_modelo.shape)
base_modelo.head()

## Célula 9 — Inspeção rápida da variável-alvo

**Objetivo**  
Verificar o balanceamento da classe.

**Por que isso é importante?**  
Se a classe de risco estiver muito desbalanceada, accuracy sozinha pode enganar.  
Por isso, neste notebook usamos também:
- balanced accuracy
- F1
- ROC AUC

In [ ]:
distribuicao_alvo = (
    base_modelo["risco_defasagem_proximo_ano"]
    .value_counts(dropna=False)
    .rename_axis("classe")
    .reset_index(name="quantidade")
)
distribuicao_alvo["percentual"] = distribuicao_alvo["quantidade"] / distribuicao_alvo["quantidade"].sum()
distribuicao_alvo

## Célula 10 — Seleção das features

**Critério usado**
Mantemos apenas variáveis disponíveis no ano `t`, isto é, antes do evento que queremos prever no ano `t+1`.

**Features utilizadas**
- acadêmicas: `ida`, `mat`, `por`, `ing`
- engajamento e atitude: `ieg`, `iaa`, `ipv`
- psicossociais: `ips`, `ipp`
- adequação e índice agregado: `ian`, `inde`
- histórico/estrutura: `idade`, `ano_ingresso`, `cg`, `cf`, `ct`

**Observação importante sobre `IAN`**
`IAN` continua sendo uma variável forte.  
Ela pode ajudar a prever, mas também exige cautela interpretativa porque está conceitualmente próxima do fenômeno da defasagem.

Por isso:
- validamos de forma mais rígida;
- regularizamos o modelo.

In [ ]:
features = [
    "idade", "ano_ingresso",
    "cg", "cf", "ct",
    "iaa", "ieg", "ips", "ipp", "ida", "ipv", "ian", "inde",
    "mat", "por", "ing"
]

features = [f for f in features if f in base_modelo.columns]

X = base_modelo[features].copy()
y = base_modelo["risco_defasagem_proximo_ano"].copy()

print("Features usadas:", features)
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

## Célula 11 — Pipeline do modelo

**O que esta célula faz?**
1. imputa valores faltantes pela mediana;
2. treina um Random Forest mais regularizado.

**O que mudou em relação ao notebook original?**
Antes o modelo estava mais flexível (`max_depth=15`), o que facilita sobreajuste.  
Agora usamos:
- `max_depth=6`
- `min_samples_leaf=8`
- `min_samples_split=20`

Isso tende a reduzir overfitting e melhorar a generalização.

In [ ]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=8,
        min_samples_split=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ))
])

## Célula 12 — Validação cruzada estratificada

**Por que usar esta etapa?**  
Ela estima o desempenho médio do modelo em diferentes divisões da amostra.

**Por que não usar apenas um único split aleatório?**  
Porque um único split pode depender muito do acaso.  
A validação cruzada dá uma visão mais estável do desempenho.

**Métricas escolhidas**
- accuracy
- balanced accuracy
- precision
- recall
- F1
- ROC AUC

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_resultados = cross_validate(
    pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

resumo_cv = pd.DataFrame({
    metrica: cv_resultados[f"test_{metrica}"]
    for metrica in scoring.keys()
})

display(resumo_cv)
print("\nMédias da validação cruzada:")
resumo_cv.mean().sort_values(ascending=False)

## Célula 13 — Teste temporal mais rigoroso

**Esta é a validação mais importante do ponto de vista de negócio.**

Aqui fazemos algo mais próximo do uso real:

- treinamos com registros de 2022 para prever a situação em 2023
- testamos com registros de 2023 para prever a situação em 2024

Em outras palavras:
o modelo aprende no passado e é testado em um período mais à frente.

Isso é mais duro e mais realista do que misturar anos aleatoriamente.

In [ ]:
# ano = ano das features (ano t)
treino_temporal = base_modelo[base_modelo["ano"] == 2022].copy()
teste_temporal  = base_modelo[base_modelo["ano"] == 2023].copy()

X_train_temp = treino_temporal[features]
y_train_temp = treino_temporal["risco_defasagem_proximo_ano"]

X_test_temp = teste_temporal[features]
y_test_temp = teste_temporal["risco_defasagem_proximo_ano"]

print("Treino temporal:", X_train_temp.shape, y_train_temp.shape)
print("Teste temporal:", X_test_temp.shape, y_test_temp.shape)

## Célula 14 — Treinamento no período passado e avaliação no período futuro

**Objetivo**  
Medir como o modelo se comporta em um cenário mais realista.

**Interpretação esperada**  
É normal o resultado aqui ser pior do que na validação cruzada.  
Isso não significa que o modelo está ruim; significa que a avaliação está mais honesta.

In [ ]:
pipeline.fit(X_train_temp, y_train_temp)

y_pred_temp = pipeline.predict(X_test_temp)
y_prob_temp = pipeline.predict_proba(X_test_temp)[:, 1]

metricas_temporais = {
    "accuracy": accuracy_score(y_test_temp, y_pred_temp),
    "balanced_accuracy": balanced_accuracy_score(y_test_temp, y_pred_temp),
    "precision": precision_score(y_test_temp, y_pred_temp, zero_division=0),
    "recall": recall_score(y_test_temp, y_pred_temp, zero_division=0),
    "f1": f1_score(y_test_temp, y_pred_temp, zero_division=0),
    "roc_auc": roc_auc_score(y_test_temp, y_prob_temp),
}

print("Métricas do teste temporal:")
for k, v in metricas_temporais.items():
    print(f"{k}: {v:.4f}")

print("\nMatriz de confusão:")
print(confusion_matrix(y_test_temp, y_pred_temp))

print("\nRelatório de classificação:")
print(classification_report(y_test_temp, y_pred_temp, zero_division=0))

## Célula 15 — Treinamento final com toda a base longitudinal disponível

**Por que treinar de novo?**  
Depois de validar a abordagem, faz sentido treinar o modelo final com toda a base longitudinal disponível (`2022→2023` e `2023→2024`) para aproveitar mais informação.

**Importante**  
Este modelo final é o que será salvo em disco.

In [ ]:
pipeline.fit(X, y)

## Célula 16 — Importância das variáveis

**Objetivo**  
Identificar quais variáveis mais contribuíram para a decisão do Random Forest.

**Cuidado interpretativo**  
Importância de variável não é causalidade.  
Ela apenas indica quais atributos o modelo mais utilizou para separar as classes.

In [ ]:
modelo_rf = pipeline.named_steps["model"]

importancias = pd.DataFrame({
    "feature": features,
    "importance": modelo_rf.feature_importances_
}).sort_values("importance", ascending=False)

importancias

## Célula 17 — Consolidação das métricas para relatório

**Objetivo**  
Guardar, em um único objeto, as métricas mais relevantes do notebook:
- médias da validação cruzada;
- métricas do teste temporal;
- lista de features;
- importâncias das variáveis.

In [ ]:
metricas_finais = {
    "validacao_cruzada_media": {
        metrica: float(resumo_cv[metrica].mean()) for metrica in resumo_cv.columns
    },
    "teste_temporal": {
        metrica: float(valor) for metrica, valor in metricas_temporais.items()
    },
    "features": features,
    "importancias": {
        row["feature"]: float(row["importance"])
        for _, row in importancias.iterrows()
    }
}

metricas_finais

## Célula 18 — Salvamento do modelo e das métricas

**Objetivo**  
Salvar o artefato do modelo para reuso futuro e registrar as métricas em JSON.

**Arquivos gerados**
- `modelo_risco_defasagem_revisado.joblib`
- `metricas_modelo_revisado.json`

In [ ]:
output_dir = Path("../models")
output_dir.mkdir(parents=True, exist_ok=True)

import joblib
joblib.dump(pipeline, output_dir / "modelo_risco_defasagem_revisado.joblib")

with open(output_dir / "metricas_modelo_revisado.json", "w", encoding="utf-8") as f:
    json.dump(metricas_finais, f, ensure_ascii=False, indent=2)

print("Arquivos salvos em:", output_dir.resolve())

## Célula 19 — Conclusões

- O modelo foi desenvolvido com foco em uma avaliação mais robusta e aderente ao objetivo preditivo proposto.
- A análise de desempenho considerou métricas complementares, como accuracy, balanced accuracy, F1 e ROC AUC, proporcionando uma visão mais abrangente da qualidade preditiva.
- A configuração mais regularizada do algoritmo busca mitigar o risco de overfitting e favorecer a capacidade de generalização para novos dados.
- Os resultados obtidos devem ser analisados de forma integrada, valorizando não apenas a acurácia, mas também o equilíbrio entre precisão, sensibilidade e discriminação entre as classes.
- Dessa forma, o modelo constitui uma base consistente para apoiar a identificação de estudantes com maior probabilidade de risco de defasagem.
